### Download, Edit, and Upload Stream Manager datasets 

In [1]:
stream_manager_config_dataset_name='Entity_Stream_Manager_Configurations'
stream_manager_function_dataset_name='Entity_Stream_Manager_Functions'

import json,os,requests
from cruciblelib import  \
    data_catalog_controller, read_controller, write_controller, \
    write_controller,authenticator,log_utils
import pandas as pd
from datetime import timezone as tz
from datetime import datetime as dt
import shutil,logging
logger = log_utils.get_logger(log_type='transformer', log_level=logging.INFO)
print()

print(' Environment Variables:')

print(' CRUCIBLE_SERVICES_HOST = ',os.getenv('CRUCIBLE_SERVICES_HOST'))
print(' CRUCIBLE_CERT_PATH = ',os.getenv('CRUCIBLE_CERT_PATH'))
print()
rc = read_controller.ReadController(version='v2')
wc = write_controller.WriteController(version='v2')
dcc = data_catalog_controller.DataCatalogController(version='v2')
auth = authenticator.Authenticator(client_id=os.environ['CRUCIBLE_CLIENT_ID'],
                                      client_secret=os.environ['CRUCIBLE_CLIENT_SECRET'],
                                      grant_type=os.environ['CRUCIBLE_GRANT_TYPE'],
                                      token_url=os.environ['CRUCIBLE_TOKEN_URL'],
                                      log_level=logging.INFO,
                                      tls_cert=os.environ['CRUCIBLE_CERT_PATH'])


config_base_dir='entity_stream_manager_config_files'
script_base_dir='entity_stream_manager_functions'

PERSPECTIVE_CONFIG_FILENAME = 'perspective_config.json'
PERSPECTIVE_CONFIG_PREFIX = 'perspective_'

# Keys that live in perspective_config.json and are shared across all
# entity transformer configs within a perspective directory.
PERSPECTIVE_CONFIG_KEYS = [
    'component_track_event_dataset',
    'component_track_head_dataset',
    'entity_dataset',
    'entity_management_event_dataset',
    'principal_track_event_dataset',
    'principal_track_head_dataset',
    'report_event_dataset',
    'report_event_with_entityId',
    'custom_functions_script_name',
    'unit_conversions_script_name',
]

# try:
#     dcc.create_dataset(stream_manager_config_dataset_name, primary_key='origin_dataset', type='ENTITY')
#     print(f"Dataset {stream_manager_config_dataset_name} created successfully.")
# except Exception as e:
#     print(f"Failed to create dataset {stream_manager_config_dataset_name}: {e}")

# try:
#     dcc.create_dataset(stream_manager_function_dataset_name, primary_key='script_name', type='ENTITY')
#     print(f"Dataset {stream_manager_function_dataset_name} created successfully.")
# except Exception as e:
#     print(f"Failed to create dataset {stream_manager_function_dataset_name}: {e}")


def write_local_functions_to_crucible(dataset_name=stream_manager_function_dataset_name):
    
    dcc.token = auth.get_token()
    wc.token = auth.get_token()
    # dcc.delete_dataset_by_name(dataset_name)
    # dcc.create_dataset(dataset_name,'script_name',type='ENTITY')
    # dcc.clear_dataset_by_name(dataset_name)


    #iterate through files in script_base_dir
    script_names_local = []
    for file_name in os.listdir(script_base_dir):
        if 'py' in file_name and not 'ipynb' in file_name:
            full_file_name = script_base_dir + '/' + file_name

            print('trying to upload ',full_file_name)
  
            config_JSON = {}
            script_name = file_name.replace('.py','')
            script_names_local.append(script_name)  
            config_JSON['script_name'] = script_name
  
            #open corresponding python file
            with open(script_base_dir+'/'+config_JSON['script_name']+'.py','r') as f:
                config_JSON['script_body'] = f.read()

            wc.upsert_by_name(dataset_name,[config_JSON])
    script_names_crucible = []
    #get list of script_names in crucible
    rc.token = auth.get_token()
    dict_list=rc.search('select * from '+dataset_name)
    for config_dict in dict_list:
        script_names_crucible.append(config_dict['script_name'])
    #compare the two lists, get list of datasets
    #  in crucible that are not in local
    #if they are not the same, delete the crucible configs
    for cruc_script in script_names_crucible:
        if cruc_script not in script_names_local:
            print('deleting config for ',cruc_script)
            wc.delete_entity_record_by_name(dataset_name,cruc_script)

def write_local_config_files_to_crucible(dataset_name=stream_manager_config_dataset_name):

    dcc.token = auth.get_token()
    wc.token = auth.get_token()
    # dcc.delete_dataset_by_name(dataset_name)
    # dcc.create_dataset(dataset_name,'origin_dataset',type='ENTITY')
    # dcc.clear_dataset_by_name(dataset_name)

    #get list of directories inside of config_base_dir, exclude files
    config_directories=os.listdir(config_base_dir)

    origin_datasets_local = []
    for dir in config_directories:
        if os.path.isdir(config_base_dir+'/'+ dir):
            perspective_dir = config_base_dir + '/' + dir

            # Upload the perspective config as its own record
            perspective_config_path = perspective_dir + '/' + PERSPECTIVE_CONFIG_FILENAME
            if os.path.exists(perspective_config_path):
                with open(perspective_config_path, 'r') as f:
                    perspective_config = json.load(f)
                origin_datasets_local.append(perspective_config['origin_dataset'])
                print(f'Uploading perspective config: {perspective_config_path}')
                wc.upsert_by_name(dataset_name, [perspective_config])
            else:
                print(f'WARNING: No {PERSPECTIVE_CONFIG_FILENAME} found in {perspective_dir}')

            # Upload each entity transformer config as its own record
            for file_name in os.listdir(perspective_dir):
                if 'json' in file_name and file_name != PERSPECTIVE_CONFIG_FILENAME:
                    full_file_name = perspective_dir + '/' + file_name
                    print('trying to upload ',full_file_name)
                    with open(full_file_name,'r') as f:
                        config_JSON = json.load(f)
                        origin_dataset = config_JSON['origin_dataset']
                        origin_datasets_local.append(origin_dataset)
                        # print('mapping:',config_JSON['origin_to_destination_mapping'])
                        wc.upsert_by_name(dataset_name,[config_JSON])
    origin_datasets_crucible = []
    #get list of origin_datasets in crucible
    rc.token = auth.get_token()
    dict_list=rc.search('select * from '+dataset_name)
    for config_dict in dict_list:
        origin_datasets_crucible.append(config_dict['origin_dataset'])
    #compare the two lists, get list of datasets   
    #  in crucible that are not in local
    #if they are not the same, delete the crucible configs
    # that aren't in local
    for cruc_dataset in origin_datasets_crucible:
        if cruc_dataset not in origin_datasets_local:
            print('deleting config for ',cruc_dataset)
            wc.delete_entity_record_by_name(dataset_name,cruc_dataset)
    
        

def clear_local_Stream_Manager_files():
    # recursively delete all files in the config directory

    # in config directory, remove subdirectories and files within them
    for root, dirs, files in os.walk(config_base_dir):
        for dir in dirs:
            shutil.rmtree(os.path.join(root, dir))

    # remove files in script subdirectory
    for root, dirs, files in os.walk(script_base_dir):
        for file in files:
            os.remove(os.path.join(root, file))

    # remove subdirectories (if needed)
    for root, dirs, files in os.walk(script_base_dir):
        for dir in dirs:
            shutil.rmtree(os.path.join(root, dir))

def download_Stream_Manager_files():

    rc.token = auth.get_token()
    dataset_name=stream_manager_config_dataset_name
    dict_list=rc.search('select * from '+dataset_name)

    for config_dict in dict_list:
        config_dict.pop('crucibleHeader',None)
        config_dict.pop("updatedTimestamp",None)
        config_dict.pop("creationTimestamp",None)
        origin_dataset = config_dict.get('origin_dataset', '')

        if origin_dataset.startswith(PERSPECTIVE_CONFIG_PREFIX):
            # This is a perspective config record
            pov = config_dict.get('perspective', origin_dataset[len(PERSPECTIVE_CONFIG_PREFIX):])
            POV_dir = config_base_dir + '/' + pov
            if not os.path.exists(POV_dir):
                os.makedirs(POV_dir)
            with open(POV_dir + '/' + PERSPECTIVE_CONFIG_FILENAME, 'w') as outfile:
                json.dump(config_dict, outfile, indent=4)
            print(f'Wrote perspective config: {POV_dir}/{PERSPECTIVE_CONFIG_FILENAME}')
        else:
            # This is an entity transformer config record
            pov = config_dict.get('perspective', 'Unknown')
            POV_dir = config_base_dir + '/' + pov
            if not os.path.exists(POV_dir):
                os.makedirs(POV_dir)
            # Strip perspective-level keys (they live in perspective_config.json)
            transformer_config = {k: v for k, v in config_dict.items() if k not in PERSPECTIVE_CONFIG_KEYS}
            with open(POV_dir + '/' + origin_dataset + '.json', 'w') as outfile:
                json.dump(transformer_config, outfile, indent=4)

    dataset_name=stream_manager_function_dataset_name
    dict_list=rc.search('select * from '+dataset_name)
    #write each config and script to a file 
    for config_dict in dict_list:
        config_dict.pop('crucibleHeader',None)
        config_dict.pop("updatedTimestamp",None)
        config_dict.pop("creationTimestamp",None)
        if not os.path.exists(script_base_dir):
            os.makedirs(script_base_dir)
        with open(script_base_dir+'/'+config_dict['script_name']+'.py', 'w') as outfile:
            outfile.write(config_dict['script_body'])

def clear_entity_config_datasets():
    try:
        dcc.clear_dataset_by_name(stream_manager_config_dataset_name)
        print(f"Dataset {stream_manager_config_dataset_name} cleared successfully.")
    except Exception as e:
        print(f"Failed to clear dataset {stream_manager_config_dataset_name}: {e}")
    try:
        dcc.clear_dataset_by_name(stream_manager_function_dataset_name)
        print(f"Dataset {stream_manager_function_dataset_name} cleared successfully.")
    except Exception as e:
        print(f"Failed to clear dataset {stream_manager_function_dataset_name}: {e}")
    # dcc.delete_dataset_by_name(stream_manager_config_dataset_name)
    # dcc.delete_dataset_by_name(stream_manager_function_dataset_name)
    # dcc.create_dataset(stream_manager_config_dataset_name, primary_key='origin_dataset', type='ENTITY')
    # dcc.create_dataset(stream_manager_function_dataset_name, primary_key='script_name', type='ENTITY')


 Environment Variables:
 CRUCIBLE_SERVICES_HOST =  crucibledev3.koverse.com
 CRUCIBLE_CERT_PATH =  /Users/cassidyta/certs/dev3ca.crt

2026-09-25 10:10:51,237:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://keycloak.crucibledev3.koverse.com/realms/crucible-services/protocol/openid-connect/token "HTTP/1.1 200 OK"
2026-09-25 10:10:51,413:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://keycloak.crucibledev3.koverse.com/realms/crucible-services/protocol/openid-connect/token "HTTP/1.1 200 OK"
2026-09-25 10:10:51,598:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://keycloak.crucibledev3.koverse.com/realms/crucible-services/protocol/openid-connect/token "HTTP/1.1 200 OK"
2026-09-25 10:10:51,761:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://keycloak.crucibledev3.koverse.com/realms/crucible-services/protocol/openid-connect/token "HTTP/1.1 200 OK"


### Download Stream Manager configuration files and functions
## STOP:
### only do this is you're certain you don't need the local config files

In [9]:
clear_local_Stream_Manager_files()
download_Stream_Manager_files()

2026-03-27 16:50:20,989:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev2.koverse.com/api/v2/read/search/dataset?allowFullTableScan=True "HTTP/2 200 OK"
Wrote perspective config: entity_stream_manager_config_files/Live/perspective_config.json
2026-03-27 16:50:21,500:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev2.koverse.com/api/v2/read/search/dataset?allowFullTableScan=True "HTTP/2 200 OK"


# STOP:
### Only do this if you're certain you don't need Crucible config files 
### Alternately, you can clear the Entity Stream Manager datasets:

In [ ]:
# clear_entity_config_datasets()
# commented to avoid accidental deletion of datasets

### ________________________________________________________________________________________________________
## STOP
### You can now edit the downloaded scripts and JSON config files in Jupyter Lab before upserting the edits
### ________________________________________________________________________________________________________

### Upload local JSON config files to dataset Stream_Manager_Configurations
#### This will overwrite configs on Crucible via upsert

In [2]:
write_local_config_files_to_crucible()

2026-09-25 10:39:18,580:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://keycloak.crucibledev3.koverse.com/realms/crucible-services/protocol/openid-connect/token "HTTP/1.1 200 OK"
Uploading perspective config: entity_stream_manager_config_files/Live_POV/perspective_config.json
2026-09-25 10:39:20,538:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev3.koverse.com/api/v2/write/entity/dataset/name/Entity_Stream_Manager_Configurations/upsert "HTTP/2 204 No Content"
2026-09-25 10:39:20,539:  INFO root write_controller Func: upsert_by_name:304-Successfully updated Entity_Stream_Manager_Configurations
trying to upload  entity_stream_manager_config_files/Live_POV/AIS_U.json
2026-09-25 10:39:21,998:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev3.koverse.com/api/v2/write/entity/dataset/name/Entity_Stream_Manager_Configurations/upsert "HTTP/2 204 No Content"
2026-09-25 10:39:21,99

### Write local Functions to dataset Stream_Manager_Functions
#### This will overwrite scripts on Crucible via upsert

In [3]:
write_local_functions_to_crucible()

trying to upload  entity_stream_manager_functions/unit_conversions.py
2026-09-25 10:39:29,474:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev3.koverse.com/api/v2/write/entity/dataset/name/Entity_Stream_Manager_Functions/upsert "HTTP/2 204 No Content"
2026-09-25 10:39:29,475:  INFO root write_controller Func: upsert_by_name:304-Successfully updated Entity_Stream_Manager_Functions
trying to upload  entity_stream_manager_functions/custom_functions.py
2026-09-25 10:39:31,067:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev3.koverse.com/api/v2/write/entity/dataset/name/Entity_Stream_Manager_Functions/upsert "HTTP/2 204 No Content"
2026-09-25 10:39:31,069:  INFO root write_controller Func: upsert_by_name:304-Successfully updated Entity_Stream_Manager_Functions
2026-09-25 10:39:31,260:  INFO httpx _client Func: _send_single_request:1025-HTTP Request: POST https://crucibledev3.koverse.com/api/v2/read/search/

### Test downloaded scripts

In [ ]:
script_name = 'unit_conversions'

######
# import script_name from Entity_Stream_Manager_Functions
######

import importlib, sys
#download and import unit_conversions.py
rc.token = auth.get_token()
dict_list=rc.search("select * from Entity_Stream_Manager_Functions")

for d in dict_list:
    if d['script_name']==script_name:
        
        script_body = d['script_body']
        module_name =  d['script_name'].replace('.py', '')
        spec = importlib.util.spec_from_loader(module_name, loader=None)
        module = importlib.util.module_from_spec(spec)
        exec(script_body, module.__dict__)
        sys.modules[module_name] = module
        globals()[module_name] = module



#####
# test the function
#####
unit_conversions.ft_to_m(1)
print('this should be about 0.3048:   ')
print('         ',unit_conversions.ft_to_m(1))



### Scratch code


#### Example here kept here kept for reference

In [ ]:
dcc.token = auth.get_token()
dcc.create_dataset_index_by_name('ComponentTrackEvents', '`crucibleHeader`.`createdDate`')

In [ ]:
# clear Entity datasets

dataset_list = ['Entities','PrincipalTrackHeads','PrincipalTrackEvents','EntityManagementEvents','ComponentTrackEvents','ComponentTrackHeads','Report_Events','Report_Events_With_Entity_ID']
dcc.token = auth.get_token()
for dataset in dataset_list:
    dcc.clear_dataset_by_name(dataset)
    # dcc.delete_dataset_by_name(dataset)


In [ ]:
# Run the four entity pipeline scripts in a local environment

perspective = 'Live'  

import subprocess
import sys
import time
import threading

scripts = [
    "entity_transformer.py",
    "entity_tracker.py",
    "entity_manager.py",
    "entity_duplicate_identifier.py",
    "entity_track_fuser.py",
]

script_dir = "../cruciblelib/cruciblelib/"
script_paths = [
    script_dir + script for script in scripts
]

def stream_subprocess_output(process, script_name):
    """Read subprocess output and print to notebook cell with script name prefix."""
    for line in iter(process.stdout.readline, b''):
        if not line:
            break
        print(f"[{script_name}] {line.decode(errors='replace').rstrip()}")
    process.stdout.close()

def stream_subprocess_error(process, script_name):
    """Read subprocess stderr and print to notebook cell with script name prefix."""
    for line in iter(process.stderr.readline, b''):
        if not line:
            break
        print(f"[{script_name}][stderr] {line.decode(errors='replace').rstrip()}")
    process.stderr.close()

processes = []
threads = []

print('Starting all four entity pipeline scripts with Live perspective...')
for script in script_paths:
    if not os.path.exists(script):
        print(f"Script {script} does not exist. Skipping...")
        continue
    # Ensure the script is executable
    if not os.access(script, os.X_OK):
        print(f"Script {script} is not executable. Please check permissions.")
        continue
    print(f'Starting {script}...')
    # Start subprocess with stdout and stderr piped
    p = subprocess.Popen(
        [sys.executable, script, perspective],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=1
    )
    processes.append(p)
    # Start threads to stream stdout and stderr to notebook cell
    t_out = threading.Thread(target=stream_subprocess_output, args=(p, script), daemon=True)
    t_err = threading.Thread(target=stream_subprocess_error, args=(p, script), daemon=True)
    t_out.start()
    t_err.start()
    threads.extend([t_out, t_err])
    time.sleep(1)  # Small delay between starts

print(f'Started {len(processes)} processes')
print('All entity pipeline scripts are now running with Live perspective')
print('Note: These processes will continue running until manually stopped')
print('To stop all processes, you may need to restart the kernel or use Ctrl+C')
print('---')
print('The output (stdout/stderr) from these scripts will now appear in this notebook cell.')

In [ ]:
# kill the processes spawned above using pkill

for script in scripts:
    script_name = script.replace('.py', '')
    print(f'Killing processes for {script_name}...')
    result = subprocess.run(['pkill', '-f', script_name], capture_output=True, text=True, check=False)
    print(f"pkill output for {script_name}:")
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)

In [ ]:
import pandas as pd
import numpy as np

# Create a DataFrame with fake data, including a boolean column
np.random.seed(42)
n_rows = 10
df = pd.DataFrame({
    'id': range(1, n_rows + 1),
    'value': np.random.randn(n_rows),
    'flag': np.random.choice([True, False], size=n_rows)
})

print(df[df['flag'] is True])


In [ ]:
set(['a','b','c']) & set(['a','b','c','d'] )